# 토큰화와 한국어 효율

같은 뜻을 담은 한국어와 영어 문장은 토큰화 방식에 따라 몇 개의 토큰이 될까요?

Python 3.12와 잠긴 패키지를 사용합니다. 첫 실행에서는 로컬 모델을 다운로드합니다.

## 두 토큰화기 준비

`AutoTokenizer.from_pretrained(model_id, revision=...)`는 모델 ID와 고정 revision을 입력받아 해당 파일을 불러오고 tokenizer 객체를 반환합니다. revision은 같은 이름의 모델 파일이 나중에 바뀌어도 이 실습에서 확인한 버전을 다시 받기 위한 식별자입니다. `vocab_size`는 tokenizer 어휘에 등록된 토큰 ID 개수를 정수로 반환합니다. 어휘 크기가 크다는 사실만으로 한국어가 더 적은 토큰으로 나뉜다고 판단할 수는 없습니다.

토큰화기의 입력은 사람이 읽는 문자열입니다. `tokenize(text)`는 토큰 문자열 목록을, `encode(text)`는 토큰 ID의 정수 목록을 반환합니다. 반환은 함수가 계산한 값을 호출한 코드에 넘긴다는 뜻입니다. 아래 셀은 두 tokenizer 객체와 각각의 어휘 크기를 출력합니다.

In [1]:
import warnings
warnings.filterwarnings("ignore", message="IProgress not found")
from transformers import AutoTokenizer

# 모델 ID와 고정 revision에 맞는 두 토큰화기를 로컬에 받아 사용합니다.
gpt2 = AutoTokenizer.from_pretrained("openai-community/gpt2", revision="607a30d783dfa663caf39e06633721c8d4cfcd7e")
ko_tokenizer = AutoTokenizer.from_pretrained("klue/bert-base", revision="77c8b3d707df785034b4e50f2da5d37be5f0f546")
print("GPT-2 어휘 크기:", gpt2.vocab_size)
print("KLUE BERT 어휘 크기:", ko_tokenizer.vocab_size)

GPT-2 어휘 크기: 50257
KLUE BERT 어휘 크기: 32000


## Ch 4 파싱 텍스트 읽기

현재 작업 폴더를 `search_paths` 목록에 먼저 넣고, 부모 폴더를 하나씩 순회하며 `append()`로 추가합니다. 다음 `for` 반복문은 이 목록에서 Ch 4 파일이 있는 경로를 찾습니다. 파일을 찾으면 `break`로 탐색을 마치고, 끝까지 찾지 못하면 오류를 발생시킵니다. 찾은 JSON의 기사 문자열을 두 토큰화기에 입력합니다.

In [2]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
import json
from pathlib import Path

search_paths = [Path.cwd()]
for parent in Path.cwd().parents:
    search_paths.append(parent)

root = None
for path in search_paths:
    candidate = path / "labs/part1/ch04-data-tools/data/article-text.json"
    if candidate.exists():
        root = path
        break
if root is None:
    raise FileNotFoundError("Ch 4 article-text.json을 찾지 못했습니다.")
handoff_path = root / "labs/part1/ch04-data-tools/data/article-text.json"
handoff = json.loads(handoff_path.read_text(encoding="utf-8"))
article_text = handoff["record"]["text"]
print("schema:", handoff["schema_version"])
print("source:", handoff["source"])
print("article:", handoff["record"]["id"], repr(article_text))
print("GPT-2:", len(gpt2.encode(article_text)), "| KLUE:", len(ko_tokenizer.tokenize(article_text)))

schema: ch04-parsed-article-v1
source: labs/part1/ch04-data-tools/data/article.html
article: A10 '골목상권 매출 회복\n주말 방문객 증가로 음식점 매출이 늘었습니다.'
GPT-2: 81 | KLUE: 17


## 한·영 문장 비교

같은 문장 쌍을 두 토큰화기의 `encode()`와 `tokenize()`에 입력합니다. `len()`은 반환된 목록의 항목 수를 셉니다. 출력은 `언어 → 토큰 수 → 실제 조각 목록` 순서입니다. 먼저 한국어 첫 문장이 GPT-2에서 몇 조각으로 나뉘는지 읽고, 같은 문장의 KLUE 결과와 비교합니다.

In [3]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
pairs = [
    ("오늘 주문 상태를 확인해 주세요.", "Please check today's order status."),
    ("환불 기준은 결제일로부터 7일입니다.", "The refund period is seven days from payment."),
]
for korean, english in pairs:
    print("\nKorean:", korean)
    print(" GPT-2:", len(gpt2.encode(korean)), gpt2.tokenize(korean))
    print(" KLUE :", len(ko_tokenizer.tokenize(korean)), ko_tokenizer.tokenize(korean))
    print("English:", english)
    print(" GPT-2:", len(gpt2.encode(english)), gpt2.tokenize(english))
    print(" KLUE :", len(ko_tokenizer.tokenize(english)), ko_tokenizer.tokenize(english))


Korean: 오늘 주문 상태를 확인해 주세요.
 GPT-2: 39 ['ì', 'ĺ', '¤', 'ë', 'Ĭ', 'ĺ', 'Ġì', '£', '¼', 'ë', '¬', '¸', 'Ġì', 'ĥ', 'ģ', 'í', 'ĥ', 'ľ', 'ë', '¥', '¼', 'Ġ', 'í', 'Ļ', 'ķ', 'ìĿ', '¸', 'íķ', '´', 'Ġì', '£', '¼', 'ì', 'Ħ', '¸', 'ì', 'ļ', 'Ķ', '.']
 KLUE : 9 ['오늘', '주문', '상태', '##를', '확인', '##해', '주세', '##요', '.']
English: Please check today's order status.
 GPT-2: 7 ['Please', 'Ġcheck', 'Ġtoday', "'s", 'Ġorder', 'Ġstatus', '.']
 KLUE : 15 ['Pl', '##ea', '##se', 'ch', '##eck', 'to', '##day', "'", 's', 'or', '##der', 'st', '##at', '##us', '.']

Korean: 환불 기준은 결제일로부터 7일입니다.
 GPT-2: 44 ['í', 'Ļ', 'ĺ', 'ë', '¶', 'Ī', 'Ġ', 'ê', '¸', '°', 'ì', '¤', 'Ģ', 'ìĿ', 'Ģ', 'Ġ', 'ê', '²', '°', 'ì', 'ł', 'ľ', 'ìĿ', '¼', 'ë', '¡', 'ľ', 'ë', '¶', 'Ģ', 'í', 'Ħ', '°', 'Ġ7', 'ìĿ', '¼', 'ì', 'ŀ', 'ħ', 'ëĭ', 'Ī', 'ëĭ', '¤', '.']
 KLUE : 11 ['환불', '기준', '##은', '결제', '##일', '##로', '##부터', '7', '##일', '##입니다', '.']
English: The refund period is seven days from payment.
 GPT-2: 9 ['The', 'Ġrefund', 'Ġperiod', 'Ġis', 'Ġsev

## 문장 길이와 토큰 수

문장 길이가 배로 늘어나면 토큰 수도 정확히 배가 되는지 확인합니다.

In [4]:
# 위 설명의 조건을 실행하고 실제 출력을 확인합니다.
base = "한국어 토큰 효율을 측정합니다."
for repeat in [1, 2, 4, 8]:
    text = " ".join([base] * repeat)
    print(repeat, "문장 | 문자", len(text), "| GPT-2", len(gpt2.encode(text)), "| KLUE", len(ko_tokenizer.tokenize(text)))

1 문장 | 문자 17 | GPT-2 37 | KLUE 8
2 문장 | 문자 35 | GPT-2 75 | KLUE 16
4 문장 | 문자 71 | GPT-2 151 | KLUE 32
8 문장 | 문자 143 | GPT-2 303 | KLUE 64


## 구조 검증

예상한 실험 조건과 출력 구조가 유지되었는지 확인합니다.

In [5]:
assert root is not None and handoff_path.exists()
assert handoff["schema_version"] == "ch04-parsed-article-v1"
assert handoff["record"]["id"] == "A10" and article_text == handoff["record"]["text"]
assert len(gpt2.encode(pairs[0][0])) > len(ko_tokenizer.encode(pairs[0][0], add_special_tokens=False))
assert len(gpt2.encode(base)) > 0
print("검증: Ch 4 파싱 텍스트와 두 토큰화기의 한·영 문장 분할을 확인했습니다.")

검증: Ch 4 파싱 텍스트와 두 토큰화기의 한·영 문장 분할을 확인했습니다.
